In [24]:

import os

In [25]:
%pwd

'f:\\Kidney-Disease-Classification-MLflow-DVC'

In [29]:
%pwd

'f:\\'

In [30]:
os.chdir("Kidney-Disease-Classification-MLflow-DVC")

In [31]:
%pwd

'f:\\Kidney-Disease-Classification-MLflow-DVC'

In [32]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [33]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [35]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )

        return prepare_base_model_config  

In [36]:
import os
import urllib.request as request
from zipfile import ZipFile
import py7zr
import tensorflow as tf


In [48]:

class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model


    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    
        
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [49]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-08-12 14:07:53,777: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-12 14:07:53,782: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-12 14:07:53,783: INFO: common: created directory at: artifacts]
[2026-08-12 14:07:53,785: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-08-12 14:07:54,347: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64

In [55]:
# it was throeing erroe so the following code was done import os

keras_models_dir = os.path.expanduser("~/.keras/models")
os.makedirs(keras_models_dir, exist_ok=True)

print(keras_models_dir)

C:\Users\Yashaswwni H R/.keras/models


In [53]:
import urllib.request
import ssl
import os

url = "https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"

save_path = os.path.expanduser(
    "~/.keras/models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
)

ssl_context = ssl._create_unverified_context()

urllib.request.urlretrieve(
    url,
    save_path,
    context=ssl_context
)

print("Downloaded successfully!")
print(save_path)

TypeError: urlretrieve() got an unexpected keyword argument 'context'

In [54]:
import urllib.request
import ssl
import os

url = "https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"

save_path = os.path.expanduser(
    "~/.keras/models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
)

os.makedirs(os.path.dirname(save_path), exist_ok=True)

ssl_context = ssl._create_unverified_context()

with urllib.request.urlopen(url, context=ssl_context) as response:
    with open(save_path, "wb") as f:
        f.write(response.read())

print("Downloaded successfully!")
print(save_path)

Downloaded successfully!
C:\Users\Yashaswwni H R/.keras/models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5


In [ ]:
import os

print(os.path.exists(
    os.path.expanduser(
        "~/.keras/models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
    )
))

True


In [ ]:
prepare_base_model.get_base_model()

[2026-08-12 14:07:09,870: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
